# Tệp: notebooks/moocradar_colab.ipynb (Bản chạy riêng trong thư mục HTTT)
# Mục đích: Tái lập bước chuẩn hóa, huấn luyện, đánh giá và lộ trình học trên Google Colab.

Notebook này được cấu hình sẵn trỏ tới `/content/drive/MyDrive/HTTT/Personalized-Learning-Path`. Tự động kiểm tra dữ liệu trong `data/raw/` (nếu chưa có bản full 152MB, bộ dữ liệu mẫu chuẩn cấu trúc MOOC-Radar đã được tạo sẵn để bạn chạy thử ngay lập tức từ đầu đến cuối).


In [16]:
from google.colab import drive
drive.mount("/content/drive")


In [17]:
# Clone mã nguồn dự án MOOC-Radar từ GitHub và đặt tên thư mục trùng khớp với PROJECT_ROOT
!git clone https://github.com/DuoChip/Personalized-Learning-Path /content/Personalized-Learning-Path

fatal: destination path '/content/Personalized-Learning-Path' already exists and is not an empty directory.


In [18]:
from pathlib import Path
import os
import subprocess

PROJECT_ROOT = Path("/content/drive/MyDrive/HTTT/Personalized-Learning-Path")
DATA_ROOT = PROJECT_ROOT / "data/raw"
PROCESSED_ROOT = PROJECT_ROOT / "data/processed"
ARTIFACT_ROOT = PROJECT_ROOT / "artifacts"
MODE = "smoke_test"  # Đổi thành "full_run" khi đã đặt bản full student-problem-fine.json vào data/raw

DATA_ROOT.mkdir(parents=True, exist_ok=True)
PROCESSED_ROOT.mkdir(parents=True, exist_ok=True)
ARTIFACT_ROOT.mkdir(parents=True, exist_ok=True)

assert (PROJECT_ROOT / "requirements.txt").exists(), "Kiểm tra lại đường dẫn PROJECT_ROOT."
assert (DATA_ROOT / "problem.json").exists(), "Thiếu problem.json trong DATA_ROOT."
assert (DATA_ROOT / "student-problem-fine.json").exists(), "Thiếu student-problem-fine.json trong DATA_ROOT."
print("Cấu hình đường dẫn và kiểm tra dữ liệu hoàn tất, hợp lệ!")
print("PROJECT_ROOT:", PROJECT_ROOT)
print("DATA_ROOT:", DATA_ROOT)


In [19]:
subprocess.run(["python", "-m", "pip", "install", "-q", "-r", str(PROJECT_ROOT / "requirements.txt")], check=True)
print("Đã cài đặt xong requirements.txt!")


In [20]:
import shutil
print("Số lõi CPU:", os.cpu_count())
print("RAM (GiB):", round(os.sysconf("SC_PHYS_PAGES") * os.sysconf("SC_PAGE_SIZE") / 2**30, 2))
print("Dung lượng trống (GiB):", round(shutil.disk_usage("/content").free / 2**30, 2))
try:
    import torch
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "không có; pipeline hỗ trợ CPU")
except ImportError:
    print("GPU check không có; pipeline hỗ trợ CPU")


## 1. Chạy kiểm tra chất lượng dữ liệu & EDA

Quét dữ liệu gốc, xuất báo cáo chất lượng và 8 biểu đồ tiếng Việt vào `artifacts/eda_<run_id>/`.


In [ ]:
env = {**os.environ, "PYTHONPATH": str(PROJECT_ROOT / "src"), "MPLCONFIGDIR": "/content/.mplconfig"}
eda_mode = "sample" if MODE == "smoke_test" else "full"
eda_cmd = ["python", "-m", "moocradar", "eda", "--data-root", str(DATA_ROOT), "--artifact-root", str(ARTIFACT_ROOT), "--mode", eda_mode]
res = subprocess.run(eda_cmd, cwd=PROJECT_ROOT, env=env, check=True, text=True, capture_output=True)
print(res.stdout)


## 2. Chuẩn bị dữ liệu (`prepare`)

Bước này đọc JSON array theo từng chuỗi, kiểm tra chất lượng và lưu các bảng chuẩn hóa (`problems.csv`, `concepts.csv`, `problem_concepts.csv`, `interactions.csv`).


In [21]:
processed_dir = PROCESSED_ROOT / ("smoke" if MODE == "smoke_test" else "full")
prepare_cmd = ["python", "-m", "moocradar", "prepare", "--data-root", str(DATA_ROOT), "--output-dir", str(processed_dir)]
if MODE == "smoke_test":
    prepare_cmd += ["--max-students", "25"]
res = subprocess.run(prepare_cmd, cwd=PROJECT_ROOT, env=env, check=True, text=True, capture_output=True)
print(res.stdout)


## 3. Huấn luyện, đánh giá và xuất hồ sơ/lộ trình (`train`)

Huấn luyện tạo thư mục run mới trong `artifacts/<run_id>/`, xuất `metrics.csv`, `predictions.csv`, `student_mastery.csv`, `cognitive_profile.csv`, `knowledge_gaps.csv`, `learning_paths.csv` và lưu tham số mô hình.


In [22]:
before = set(ARTIFACT_ROOT.glob("*/manifest.json"))
res = subprocess.run(["python", "-m", "moocradar", "train", "--processed-dir", str(processed_dir), "--artifact-root", str(ARTIFACT_ROOT), "--seed", "42"], cwd=PROJECT_ROOT, env=env, check=True, text=True, capture_output=True)
print(res.stdout)
new_manifests = set(ARTIFACT_ROOT.glob("*/manifest.json")) - before
assert len(new_manifests) == 1
run_dir = next(iter(new_manifests)).parent
print("Run đã lưu:", run_dir)


In [23]:
import csv
import json
import pandas as pd
from IPython.display import display

manifest = json.loads((run_dir / "manifest.json").read_text())
assert manifest["status"] == "complete"
print("=== KẾT QUẢ ĐÁNH GIÁ MÔ HÌNH (metrics.csv) ===")
display(pd.read_csv(run_dir / "metrics.csv"))

print("=== MẪU HỒ SƠ NĂNG LỰC (student_mastery.csv) ===")
display(pd.read_csv(run_dir / "student_mastery.csv").head(10))

print("=== MẪU LỘ TRÌNH HỌC TẬP ĐỀ XUẤT (learning_paths.csv) ===")
display(pd.read_csv(run_dir / "learning_paths.csv").head(10))


## 4. Demo suy luận JSON cho 1 sinh viên (`demo`)

Chọn một sinh viên từ bảng tương tác đã lưu và cập nhật trạng thái tức thời mà không cần train lại.


In [24]:
with (processed_dir / "interactions.csv").open() as f:
    user_id = next(csv.DictReader(f))["user_id"]
res = subprocess.run(["python", "-m", "moocradar", "demo", "--run-dir", str(run_dir), "--processed-dir", str(processed_dir), "--user-id", user_id], cwd=PROJECT_ROOT, env=env, check=True, text=True, capture_output=True)
print(res.stdout)
print("Chia sẻ thư mục kết quả Drive này:", run_dir)


## 5. Đồng bộ Git: Tạo nhánh, Commit, Push và Merge lên GitHub (`DuoChip/Personalized-Learning-Path`)

Điền GitHub Personal Access Token (`GITHUB_TOKEN`) của bạn vào biến bên dưới (hoặc lưu trong Colab Secrets với tên `GITHUB_TOKEN`) rồi chạy cell để tự động đồng bộ toàn bộ mã nguồn mới nhất từ `HTTT/Personalized-Learning-Path` sang nhánh `feat/personalized-smoothed-baseline`, push lên GitHub và merge vào nhánh `main`.


In [ ]:
import shutil
import subprocess
from pathlib import Path
try:
    from google.colab import userdata
    GITHUB_TOKEN = userdata.get("GITHUB_TOKEN")
except Exception:
    GITHUB_TOKEN = ""  # <-- Hoặc dán trực tiếp GitHub Personal Access Token (ghp_...) vào đây

BRANCH_NAME = "feat/personalized-smoothed-baseline"
GIT_REPO_DIR = Path("/content/drive/MyDrive/Personalized-Learning-Path")
HTTT_REPO_DIR = Path("/content/drive/MyDrive/HTTT/Personalized-Learning-Path")

# 1. Đồng bộ toàn bộ code đã nâng cấp từ HTTT sang thư mục Git repo
for sub in ["src/moocradar", "notebooks", "configs", "docs", "tests"]:
    src_sub = HTTT_REPO_DIR / sub
    dst_sub = GIT_REPO_DIR / sub
    dst_sub.mkdir(parents=True, exist_ok=True)
    for f in src_sub.glob("*"):
        if f.is_file():
            shutil.copy2(f, dst_sub / f.name)

subprocess.run(["git", "config", "user.name", "thucpnjob"], cwd=GIT_REPO_DIR, check=True)
subprocess.run(["git", "config", "user.email", "thucphungngoc@gmail.com"], cwd=GIT_REPO_DIR, check=True)

# 2. Tạo nhánh mới và commit các thay đổi
subprocess.run(["git", "checkout", "-B", BRANCH_NAME], cwd=GIT_REPO_DIR, check=True)
subprocess.run(["git", "add", "src/", "notebooks/", "configs/", "docs/", "tests/", "README.md", "requirements.txt"], cwd=GIT_REPO_DIR, check=True)
subprocess.run(["git", "commit", "-m", "feat: upgrade per-student smoothed concept baseline, ZPD recommendation, and NCDM module"], cwd=GIT_REPO_DIR, check=False)

# 3. Merge vào main và push lên GitHub nếu có GITHUB_TOKEN
subprocess.run(["git", "checkout", "main"], cwd=GIT_REPO_DIR, check=True)
subprocess.run(["git", "merge", "--no-ff", BRANCH_NAME, "-m", f"Merge branch {BRANCH_NAME} into main"], cwd=GIT_REPO_DIR, check=False)

if GITHUB_TOKEN:
    remote_url = f"https://{GITHUB_TOKEN}@github.com/DuoChip/Personalized-Learning-Path.git"
    subprocess.run(["git", "push", remote_url, BRANCH_NAME], cwd=GIT_REPO_DIR, check=True)
    subprocess.run(["git", "push", remote_url, "main"], cwd=GIT_REPO_DIR, check=True)
    print("✅ Đã push nhánh", BRANCH_NAME, "và merge vào main thành công lên GitHub!")
else:
    print("✅ Đã tạo nhánh", BRANCH_NAME, "và merge vào main cục bộ trên Drive! Hãy điền GITHUB_TOKEN để push lên GitHub.")
